In [1]:
import pandas as pd
from ast import literal_eval
from math import radians, sin, cos, sqrt, atan2
from sklearn.cluster import KMeans
import numpy as np

In [4]:
# 将字符串类型的列表转换为真正的列表对象
def safe_literal_eval(s):
    try:
        return literal_eval(s.replace(';', ','))  # 处理可能存在的分号分隔符
    except:
        return []

In [6]:
df1 = pd.read_csv("/home/users/lichuan/shared/gcj_trips_150103.jld2_trips.csv")
df2 = pd.read_csv("/home/users/lichuan/shared/gcj_trips_150104.jld2_trips.csv")
df3 = pd.read_csv("/home/users/lichuan/shared/gcj_trips_150105.jld2_trips.csv")
df4 = pd.read_csv("/home/users/lichuan/shared/gcj_trips_150106.jld2_trips.csv")
df5 = pd.read_csv("/home/users/lichuan/shared/gcj_trips_150107.jld2_trips.csv")
dataframes = [df1, df2, df3, df4, df5]
df = pd.concat(dataframes, ignore_index=True)
# 转换 tms 列
df["lon"] = df["lon"].apply(safe_literal_eval)
df["lat"] = df["lat"].apply(safe_literal_eval)
df["begin_lon"] = df["lon"].apply(lambda x: x[0] if len(x) > 0 else None)
df["begin_lat"] = df["lat"].apply(lambda x: x[0] if len(x) > 0 else None)
df["end_lon"] = df["lon"].apply(lambda x: x[len(x)-1] if len(x) > 0 else None)
df["end_lat"] = df["lat"].apply(lambda x: x[len(x)-1] if len(x) > 0 else None)
df["begin_position"] = df.apply(
    lambda row: [row["lon"][0], row["lat"][0]] 
    if len(row["lon"]) > 0 and len(row["lat"]) > 0 
    else None,
    axis=1
)

# 生成 end_position 列（同理）
df["end_position"] = df.apply(
    lambda row: [row["lon"][-1], row["lat"][-1]] 
    if len(row["lon"]) > 0 and len(row["lat"]) > 0 
    else None,
    axis=1
)

In [7]:
def haversine_distance(lon1, lat1, lon2, lat2):
    """计算两点直线距离

    Args:
        lon1 (float): 经度1
        lat1 (float): 纬度1
        lon2 (float): 经度2
        lat2 (float): 纬度2

    Returns:
        float: 直线距离
    """
    # 将角度转换为弧度
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    
    # 差值
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    # Haversine 公式
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    # 地球半径（单位：千米）
    R = 6378.137
    distance = R * c
    
    return distance  # 返回两点间的距离（单位：千米）
  
  
def calculate_driving_distance(lon_list, lat_list):
    if len(lon_list) < 2 or len(lat_list) < 2:
        return 0.0
    
    total_distance = 0.0
    for i in range(len(lon_list)-1):
        lat1, lon1 = float(lat_list[i]), float(lon_list[i])
        lat2, lon2 = float(lat_list[i+1]), float(lon_list[i+1])
        total_distance += haversine_distance(lat1, lon1, lat2, lon2)
    
    return round(total_distance, 4)  # 保留4位小数
    
    
def api14_average_driving_distance_and_time():
    # 读取 CSV 文件
    df = pd.read_csv("./example.csv")

    # 转换 tms 列
    df["tms"] = df["tms"].apply(safe_literal_eval)
    df["lon"] = df["lon"].apply(safe_literal_eval)
    df["lat"] = df["lat"].apply(safe_literal_eval)

    # 新增 begin_time 列（取 tms 列表的第一个元素）
    df["begin_time"] = df["tms"].apply(lambda x: x[0] if len(x) > 0 else None)
    df["end_time"] = df["tms"].apply(lambda x: x[len(x)-1] if len(x) > 0 else None)
    df["during_time"] = (df["end_time"] - df["begin_time"])/60
    df["driving_distance_km"] = df.apply(
        lambda row: calculate_driving_distance(row["lon"], row["lat"]), 
        axis=1
    )
    df["average_speed_km_per_hour"] = df["driving_distance_km"] / (df["during_time"]/60)
    
    return (df["driving_distance_km"].sum() / len(df["driving_distance_km"]), df["during_time"].sum()/len(df["during_time"]))


def add_cluster_id(df, k=5):
    """
    为 DataFrame 新增上车点和下车点的聚类 ID 列
    :param df: 输入 DataFrame, 需包含 'begin_position' 和 'end_position' 列
    :param k: 聚类簇数
    :return: 新增了聚类 ID 列的 DataFrame
    """
    # 提取有效上车点和下车点
    begin_valid = df['begin_position'].dropna()
    end_valid = df['end_position'].dropna()
    
    # 检查是否有足够数据
    if len(begin_valid) + len(end_valid) < k:
        df['begin_cluster_id'] = 0
        df['end_cluster_id'] = 0
        return df
    
    # 合并所有有效坐标
    all_positions = begin_valid.tolist() + end_valid.tolist()
    coordinates = np.array(all_positions)
    
    # K-means 聚类
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(coordinates)
    
    # 生成簇标签（从1开始，0保留给无效数据）
    labels = kmeans.labels_ + 1
    
    # 分割标签为上车站点和下车站点
    begin_labels = labels[:len(begin_valid)]
    end_labels = labels[len(begin_valid):]
    
    # 创建新列并填充簇ID
    df['begin_cluster_id'] = 0  # 默认0表示无效位置
    df['end_cluster_id'] = 0    # 默认0表示无效位置
    
    # 为有效位置分配聚类ID
    df.loc[begin_valid.index, 'begin_cluster_id'] = begin_labels
    df.loc[end_valid.index, 'end_cluster_id'] = end_labels
    
    # 计算簇中心点（使用KMeans自带的中心点）
    cluster_centers = kmeans.cluster_centers_
    center_map = {i+1: center for i, center in enumerate(cluster_centers)}
    
    # 添加簇中心点列
    df['begin_cluster_center'] = df['begin_cluster_id'].map(
        lambda x: center_map.get(x, np.nan)
    )
    df['end_cluster_center'] = df['end_cluster_id'].map(
        lambda x: center_map.get(x, np.nan)
    )
    
    df["begin_cluster_center_lon"] = df["begin_cluster_center"].apply(lambda x: x[0] if len(x) > 0 else None)
    df["begin_cluster_center_lat"] = df["begin_cluster_center"].apply(lambda x: x[1] if len(x) > 0 else None)
    df["end_cluster_center_lon"] = df["end_cluster_center"].apply(lambda x: x[0] if len(x) > 0 else None)
    df["end_cluster_center_lat"] = df["end_cluster_center"].apply(lambda x: x[1] if len(x) > 0 else None)
    
    return df

In [8]:
df = add_cluster_id(df, 8)

In [9]:
df.columns.tolist()

['wgs_lon',
 'wgs_lat',
 'tms',
 'devid',
 'roads',
 'time',
 'frac',
 'route',
 'route_heading',
 'route_geom',
 'lon',
 'lat',
 'begin_lon',
 'begin_lat',
 'end_lon',
 'end_lat',
 'begin_position',
 'end_position',
 'begin_cluster_id',
 'end_cluster_id',
 'begin_cluster_center',
 'end_cluster_center',
 'begin_cluster_center_lon',
 'begin_cluster_center_lat',
 'end_cluster_center_lon',
 'end_cluster_center_lat']

In [13]:
df['tms'].head(2)

0    [1.4202432e9, 1.42024323e9, 1.42024329e9, 1.42...
1    [1.4202432e9, 1.420243301e9, 1.420243361e9, 1....
Name: tms, dtype: object

In [14]:
df['begin_cluster_center'].head(2)

0     [126.67380285202498, 45.73016690136611]
1    [126.62187549175023, 45.703193569646565]
Name: begin_cluster_center, dtype: object

In [23]:
cluster_count = 8
cluster_centers = {cid+1:[] for cid in range(cluster_count)}
temp_cnt = 0
for _, row in df.iterrows():
    cluster_centers[row['begin_cluster_id']] = row['begin_cluster_center'].tolist()
    cluster_centers[row['end_cluster_id']] = row['end_cluster_center'].tolist()
    temp_cnt += 1
    if temp_cnt > 30:
        break
cluster_centers

{1: [126.7248004537484, 45.74353229297238],
 2: [126.62248935192684, 45.75353537991027],
 3: [126.69472427879992, 45.77618803113608],
 4: [126.5603597682078, 45.70655618629882],
 5: [126.67380285202498, 45.73016690136611],
 6: [126.65429423404818, 45.77189680658441],
 7: [126.62187549175023, 45.703193569646565],
 8: [126.58721477363214, 45.75452435984508]}

In [27]:
from datetime import datetime, timedelta

def convert_timestamp_list(timestamp_list):
    """
    将时间戳列表转换为UTC+8时间对象列表
    :param timestamp_list: 时间戳列表
    :return: UTC+8时间对象列表
    """
    beijing_times = []
    for ts in timestamp_list:
        # 处理毫秒级时间戳
        if ts > 1e12:
            ts_seconds = ts / 1000.0
        else:
            ts_seconds = ts
        
        # 转换为UTC时间
        try:
            utc_time = datetime.fromtimestamp(ts_seconds)
            # 转换为北京时间
            beijing_time = utc_time + timedelta(hours=8)
            beijing_times.append(beijing_time)
        except (ValueError, OSError):
            # 处理无效时间戳（如负值或过大值）
            beijing_times.append(None)
    
    return beijing_times

# 方法1：使用apply函数（推荐）
df['beijing_time'] = df['tms'].apply(convert_timestamp_list)
df['beijing_time'].head()

0    [2015-01-03 03:00:00, 2015-01-03 03:00:30, 201...
1    [2015-01-03 03:00:00, 2015-01-03 03:01:41, 201...
2    [2015-01-03 03:00:00, 2015-01-03 03:00:30, 201...
3    [2015-01-03 03:00:00, 2015-01-03 03:00:51, 201...
4    [2015-01-03 03:00:00, 2015-01-03 03:00:30, 201...
Name: beijing_time, dtype: object

In [28]:
from sklearn.neighbors import BallTree
from collections import defaultdict
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# 准备聚类中心坐标
centers = np.array([cluster_centers[cid] for cid in range(1, cluster_count+1)])
centers_rad = np.radians(centers)
tree = BallTree(centers_rad, metric='haversine')

# 初始化按日期和小时统计的拥挤度字典
# 新结构: {cluster_id: {date: {hour: count}}}
crowding_by_cluster = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(int)
    )
)

# 2. 提取日期字符串（格式：YYYY-MM-DD）
df['date_str'] = df['beijing_time'].apply(
    lambda dt_list: [dt.strftime('%Y-%m-%d') for dt in dt_list]
)

# 遍历每条轨迹
for _, row in df.iterrows():
    # 提取当前轨迹的所有点
    points = list(zip(row['lon'], row['lat']))
    
    # 转换为numpy数组
    points_arr = np.array(points)
    points_rad = np.radians(points_arr)
    
    # 批量查询最近的聚类中心
    distances, indices = tree.query(points_rad, k=1)
    distances_km = distances * 6371  # 转换为公里
    
    # 处理每个点
    for i in range(len(points)):
        # 获取最近聚类中心的ID
        nearest_cid = indices[i][0] + 1
        
        # 检查距离是否在2公里范围内
        if distances_km[i][0] <= 2:
            # 提取该点的日期和时间
            date_str = row['date_str'][i]
            hour = row['beijing_time'][i].hour
            
            # 更新该聚类ID、日期和小时的拥挤度计数
            crowding_by_cluster[nearest_cid][date_str][hour] += 1

# 组织结果：按cluster_id分组，然后按日期分组
final_results = {}
for cid in range(1, cluster_count+1):
    cluster_data = crowding_by_cluster.get(cid, {})
    
    # 创建该聚类ID的结果结构
    cluster_result = {}
    
    # 只取前5天
    date_keys = sorted(cluster_data.keys())[:5] if cluster_data else []
    
    for date_str in date_keys:
        # 创建该日期24小时的完整结构
        day_data = {}
        for hour in range(24):
            day_data[hour] = cluster_data[date_str].get(hour, 0)
        
        cluster_result[date_str] = day_data
    
    final_results[cid] = cluster_result

# 输出最终结果
for cid, cluster_data in final_results.items():
    print(f"\nCluster {cid}:")
    for date_str, day_data in cluster_data.items():
        print(f"  Date: {date_str}")
        for hour in range(24):
            count = day_data[hour]
            if count > 0:  # 只显示有数据的点
                print(f"    Hour {hour:02d}: {count}")


Cluster 1:
  Date: 2015-01-03
    Hour 03: 2656
    Hour 04: 1648
    Hour 05: 955
    Hour 06: 805
    Hour 07: 757
    Hour 08: 2027
    Hour 09: 3738
    Hour 10: 8225
    Hour 11: 7305
    Hour 12: 7218
    Hour 13: 7480
    Hour 14: 7056
    Hour 15: 6795
    Hour 16: 8033
    Hour 17: 9805
    Hour 18: 10314
    Hour 19: 10092
    Hour 20: 9340
    Hour 21: 8143
    Hour 22: 7303
    Hour 23: 8133
  Date: 2015-01-04
    Hour 00: 7302
    Hour 01: 5342
    Hour 02: 3432
    Hour 03: 2751
    Hour 04: 1412
    Hour 05: 884
    Hour 06: 892
    Hour 07: 992
    Hour 08: 2071
    Hour 09: 4562
    Hour 10: 9376
    Hour 11: 9234
    Hour 12: 8130
    Hour 13: 12836
    Hour 14: 13843
    Hour 15: 15931
    Hour 16: 15804
    Hour 17: 17999
    Hour 18: 19402
    Hour 19: 20889
    Hour 20: 19150
    Hour 21: 18233
    Hour 22: 13701
    Hour 23: 13731
  Date: 2015-01-05
    Hour 00: 14157
    Hour 01: 10764
    Hour 02: 7048
    Hour 03: 4960
    Hour 04: 2964
    Hour 05: 1995
    

In [30]:
from sklearn.neighbors import BallTree
from collections import defaultdict
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# 初始化活跃度字典 - 结构: {cluster_id: {date: {hour: count}}}
activity_by_cluster = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(int)
    )
)

df["end_beijing_time"] = df["beijing_time"].apply(lambda x: x[len(x)-1] if len(x) > 0 else None)

# 2. 提取结束日期字符串
df['end_date_str'] = df['end_beijing_time'].apply(
    lambda dt: dt.strftime('%Y-%m-%d')
)

# 遍历每条轨迹（只处理下车点）
for _, row in df.iterrows():
    # 获取结束点坐标
    end_lon, end_lat = row['end_position']
    point = np.array([[end_lon, end_lat]])
    
    # 转换为弧度
    point_rad = np.radians(point)
    
    # 查询最近的聚类中心
    distance, index = tree.query(point_rad, k=1)
    distance_km = distance[0][0] * 6371  # 转换为公里
    
    # 检查距离是否在2公里范围内
    if distance_km <= 2:
        # 获取聚类ID
        cid = index[0][0] + 1
        
        # 获取结束时间和日期
        hour = row['end_beijing_time'].hour
        date_str = row['end_date_str']
        
        # 更新活跃度计数
        activity_by_cluster[cid][date_str][hour] += 1

# 组织结果：按cluster_id分组，然后按日期分组
final_activity = {}
for cid in range(1, cluster_count+1):
    cluster_data = activity_by_cluster.get(cid, {})
    
    # 创建该聚类ID的结果结构
    cluster_result = {}
    
    # 只取前5天
    date_keys = sorted(cluster_data.keys())[:5] if cluster_data else []
    
    for date_str in date_keys:
        # 创建该日期24小时的完整结构
        day_data = {}
        for hour in range(24):
            day_data[hour] = cluster_data[date_str].get(hour, 0)
        
        cluster_result[date_str] = day_data
    
    final_activity[cid] = cluster_result

# 输出最终结果
for cid, cluster_data in final_activity.items():
    print(f"\nCluster {cid} Activity:")
    for date_str, day_data in cluster_data.items():
        print(f"  Date: {date_str}")
        for hour in range(24):
            count = day_data[hour]
            if count > 0:  # 只显示有数据的点
                print(f"    Hour {hour:02d}: {count}")


Cluster 1 Activity:
  Date: 2015-01-03
    Hour 03: 68
    Hour 04: 55
    Hour 05: 37
    Hour 06: 28
    Hour 07: 27
    Hour 08: 67
    Hour 09: 108
    Hour 10: 181
    Hour 11: 187
    Hour 12: 178
    Hour 13: 259
    Hour 14: 214
    Hour 15: 228
    Hour 16: 255
    Hour 17: 305
    Hour 18: 334
    Hour 19: 252
    Hour 20: 270
    Hour 21: 211
    Hour 22: 196
    Hour 23: 183
  Date: 2015-01-04
    Hour 00: 181
    Hour 01: 146
    Hour 02: 134
    Hour 03: 78
    Hour 04: 45
    Hour 05: 29
    Hour 06: 41
    Hour 07: 34
    Hour 08: 58
    Hour 09: 106
    Hour 10: 231
    Hour 11: 301
    Hour 12: 275
    Hour 13: 354
    Hour 14: 383
    Hour 15: 449
    Hour 16: 467
    Hour 17: 529
    Hour 18: 614
    Hour 19: 569
    Hour 20: 585
    Hour 21: 475
    Hour 22: 354
    Hour 23: 294
  Date: 2015-01-05
    Hour 00: 327
    Hour 01: 284
    Hour 02: 248
    Hour 03: 145
    Hour 04: 99
    Hour 05: 66
    Hour 06: 61
    Hour 07: 61
    Hour 08: 133
    Hour 09: 173
   

In [ ]:
cluster_centers
# 会展、爱建、博物馆、西城红场、三大动力路

{1: [126.7248004537484, 45.74353229297238],
 2: [126.62248935192684, 45.75353537991027],
 3: [126.69472427879992, 45.77618803113608],
 4: [126.5603597682078, 45.70655618629882],
 5: [126.67380285202498, 45.73016690136611],
 6: [126.65429423404818, 45.77189680658441],
 7: [126.62187549175023, 45.703193569646565],
 8: [126.58721477363214, 45.75452435984508]}

In [32]:
total_activity = {cid+1:0 for cid in range(cluster_count)}
for cid, cluster_data in final_activity.items():
    for date_str, day_data in cluster_data.items():
        for hour in range(24):
            count = day_data[hour]
            if count > 0:  # 只显示有数据的点
                total_activity[cid] += count

total_crowding = {cid+1:0 for cid in range(cluster_count)}
for cid, cluster_data in final_results.items():
    for date_str, day_data in cluster_data.items():
        for hour in range(24):
            count = day_data[hour]
            if count > 0:  # 只显示有数据的点
                total_crowding[cid] += count

print(total_activity)
print(total_crowding)

{1: 32315, 2: 196013, 3: 121937, 4: 24611, 5: 144529, 6: 183614, 7: 89594, 8: 87544}
{1: 1179511, 2: 10354491, 3: 4556295, 4: 1010170, 5: 6053124, 6: 8492509, 7: 3202465, 8: 3833589}


In [33]:
day_activity = {cid+1:{day:0 for day in range(3,8)} for cid in range(cluster_count)}
for cid, cluster_data in final_activity.items():
    for date_str, day_data in cluster_data.items():
        for hour in range(24):
            count = day_data[hour]
            if count > 0:  # 只显示有数据的点
                day_activity[cid][int(date_str[len(date_str)-1])] += count
day_crowding = {cid+1:{day:0 for day in range(3,8)} for cid in range(cluster_count)}
for cid, cluster_data in final_results.items():
    for date_str, day_data in cluster_data.items():
        for hour in range(24):
            count = day_data[hour]
            if count > 0:  # 只显示有数据的点
                day_crowding[cid][int(date_str[len(date_str)-1])] += count

In [34]:
print(day_activity)
print(day_crowding)

{1: {3: 3643, 4: 6732, 5: 7794, 6: 7234, 7: 6912}, 2: {3: 24669, 4: 38340, 5: 47226, 6: 44185, 7: 41593}, 3: {3: 12831, 4: 23827, 5: 29610, 6: 29062, 7: 26607}, 4: {3: 3122, 4: 5074, 5: 5657, 6: 5688, 7: 5070}, 5: {3: 14047, 4: 28992, 5: 34490, 6: 35298, 7: 31702}, 6: {3: 20592, 4: 35480, 5: 44992, 6: 42729, 7: 39821}, 7: {3: 11044, 4: 17880, 5: 22021, 6: 20122, 7: 18527}, 8: {3: 10827, 4: 17623, 5: 20779, 6: 19997, 7: 18318}}
{1: {3: 127828, 4: 237899, 5: 272513, 6: 269164, 7: 272107}, 2: {3: 1156058, 4: 1907356, 5: 2367170, 6: 2354026, 7: 2569881}, 3: {3: 454871, 4: 855223, 5: 1065253, 6: 1067853, 7: 1113095}, 4: {3: 114238, 4: 194416, 5: 223686, 6: 235105, 7: 242725}, 5: {3: 591508, 4: 1159840, 5: 1403627, 6: 1417539, 7: 1480610}, 6: {3: 880097, 4: 1525208, 5: 1981410, 6: 1965640, 7: 2140154}, 7: {3: 338864, 4: 605936, 5: 750903, 6: 731550, 7: 775212}, 8: {3: 414141, 4: 725884, 5: 874274, 6: 887785, 7: 931505}}


In [37]:
import pandas as pd
from datetime import datetime, timedelta

def create_activity_csv(data_dict, output_file):
    # 准备存储所有数据的列表
    rows = []
    
    # 获取所有日期并排序
    all_dates = sorted({date for cluster_data in data_dict.values() for date in cluster_data.keys()})
    
    for cluster_id, cluster_data in data_dict.items():
        # 对每个集群的日期排序
        sorted_dates = sorted(cluster_data.keys())
        
        for i, date in enumerate(sorted_dates):
            # 获取前一天日期
            prev_date = (datetime.strptime(date, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
            
            # 获取当日活跃度数据
            today_activity = cluster_data[date]
            
            # 获取昨日活跃度数据（如果存在）
            if i > 0 and prev_date in cluster_data:
                yesterday_activity = cluster_data[prev_date]
            else:
                yesterday_activity = [0] * 24  # 第一天或前一天不存在，设为0
                
            # 添加每个小时的数据
            for hour in range(24):
                rows.append({
                    "cluster_id": cluster_id,
                    "date": date,
                    "hour": hour,
                    "activity": today_activity[hour],
                    "yesterday_activity": yesterday_activity[hour]
                })
    
    # 创建DataFrame并保存为CSV
    df = pd.DataFrame(rows)
    df.to_csv(output_file, index=False)
    print(f"CSV文件已保存至: {output_file}")

# 使用示例（假设data_dict已经存在）
create_activity_csv(final_activity, "cluster_activity.csv")

CSV文件已保存至: cluster_activity.csv


In [42]:
import pandas as pd
import csv

def create_day_activity_csv(day_activity, output_file):
    """
    将 day_activity 字典转换为 CSV 文件
    
    参数:
    day_activity (dict): {cluster_id: {day_id: day_activity}}
    output_file (str): 输出的 CSV 文件路径
    """
    # 准备数据
    data = []
    
    # 遍历所有商圈
    for cluster_id, day_data in day_activity.items():
        # 对日期进行排序（day1, day2, ..., day5）
        sorted_days = sorted(day_data.keys())
        
        for day_id in sorted_days:
            activity = day_data[day_id]
            data.append({
                'cluster_id': cluster_id,
                'day_id': '2015010' + str(day_id),
                'activity': activity
            })
    
    # 创建 DataFrame 并保存为 CSV
    df = pd.DataFrame(data)
    df.to_csv(output_file, index=False)
    print(f"CSV文件已保存至: {output_file}")

# 使用示例
# 假设 day_activity 是这样的结构:
# day_activity = {
#     1: {'day1': 1500, 'day2': 1600, 'day3': 1700, 'day4': 1800, 'day5': 1900},
#     2: {'day1': 2000, 'day2': 2100, 'day3': 2200, 'day4': 2300, 'day5': 2400},
#     # ... 其他商圈
# }

# 调用函数生成 CSV
create_day_activity_csv(day_activity, "day_activity.csv")

CSV文件已保存至: day_activity.csv


In [43]:
create_day_activity_csv(day_crowding, "day_crowding.csv")

CSV文件已保存至: day_crowding.csv


In [46]:
mall_shopping_df = pd.read_csv("./mall_shopping.csv")
mall_shopping_df.head()

,Unnamed: 0,lon,lat,begin_lon,begin_lat,end_lon,end_lat,begin_area_id,end_area_id,begin_cluster_center_lon,begin_cluster_center_lat,end_cluster_center_lon,end_cluster_center_lat,crowding_degree,crowding_score,activity_level,activity_score
0,0,"[126.69578864326199, 126.69596393816752, 126.6...","[45.74767693533434, 45.74653155382219, 45.7463...",126.695789,45.747677,126.624987,45.766095,1,3,126.707423,45.757364,126.607516,45.753799,8489803,9.414975,284693,7.743240
1,1,"[126.63120997299498, 126.63135521073364, 126.6...","[45.663033752434735, 45.66303091339197, 45.662...",126.631210,45.663034,126.631561,45.662927,5,4,126.587142,45.708719,126.639282,45.706251,3022770,2.999980,185474,3.849568
2,2,"[126.75887304173547, 126.75887304239316, 126.7...","[45.75062279543633, 45.75062679436818, 45.7503...",126.758873,45.750623,126.705126,45.731286,1,1,126.707423,45.757364,126.701815,45.749967,4040202,4.193831,274303,7.335503
3,3,"[126.68779506405106, 126.68787022159448, 126.6...","[45.70938589731701, 45.70903612240171, 45.7086...",126.687795,45.709386,126.684533,45.722169,3,1,126.662821,45.721350,126.701815,45.749967,4040202,4.193831,274303,7.335503
4,4,"[126.65520537035115, 126.65633838574469, 126.6...","[45.75066455471406, 45.75126635101695, 45.7524...",126.655205,45.750665,126.657919,45.765755,4,2,126.658150,45.772086,126.656810,45.766121,8988377,10.000000,342200,10.000000


In [49]:
import csv

# 哈尔滨的40个景点/饭店/地标数据
places = [
    {"名称": "圣索菲亚大教堂", "简介": "哈尔滨最具代表性的拜占庭式建筑，俄罗斯文化象征", "地址": "哈尔滨市道里区透笼街88号"},
    {"名称": "中央大街", "简介": "百年历史俄式风情步行街，哈尔滨的城市名片", "地址": "哈尔滨市道里区中央大街"},
    {"名称": "太阳岛风景区", "简介": "国家5A级旅游景区，夏季避暑胜地，冬季雪博会主场", "地址": "哈尔滨市松北区太阳大道1号"},
    {"名称": "哈尔滨冰雪大世界", "简介": "世界最大冰雪主题乐园，每年冬季开放", "地址": "哈尔滨市松北区太阳岛西侧"},
    {"名称": "哈尔滨极地馆", "简介": "中国首家极地演艺游乐园，有著名的白鲸表演", "地址": "哈尔滨市松北区太阳大道3号"},
    {"名称": "防洪纪念塔", "简介": "为纪念1957年哈尔滨人民战胜特大洪水而建", "地址": "哈尔滨市道里区中央大街尽头"},
    {"名称": "龙塔", "简介": "亚洲第一高钢塔，哈尔滨地标性建筑", "地址": "哈尔滨市南岗区长江路178号"},
    {"名称": "老道外中华巴洛克", "简介": "中国最大巴洛克建筑群，百年历史文化街区", "地址": "哈尔滨市道外区南二道街"},
    {"名称": "哈尔滨大剧院", "简介": "世界级建筑杰作，中国最美剧院之一", "地址": "哈尔滨市松北区文化中心岛"},
    {"名称": "伏尔加庄园", "简介": "俄式风情主题庄园，重现俄罗斯经典建筑", "地址": "哈尔滨市香坊区哈成路16公里处"},
    {"名称": "东北虎林园", "简介": "世界最大东北虎繁育基地", "地址": "哈尔滨市松北区松北街88号"},
    {"名称": "哈尔滨博物馆", "简介": "展示哈尔滨历史文化的综合性博物馆", "地址": "哈尔滨市道里区柳树街13号"},
    {"名称": "哈尔滨工业大学", "简介": "著名985高校，国防七子之一", "地址": "哈尔滨市南岗区西大直街92号"},
    {"名称": "黑龙江省博物馆", "简介": "展示黑龙江历史与自然资源的省级博物馆", "地址": "哈尔滨市南岗区红军街50号"},
    {"名称": "哈尔滨音乐公园", "简介": "以音乐为主题的城市公园", "地址": "哈尔滨市道里区友谊西路"},
    {"名称": "斯大林公园", "简介": "松花江畔开放式公园，俄式风格建筑", "地址": "哈尔滨市道里区斯大林街3号"},
    {"名称": "哈尔滨工程大学", "简介": "著名211高校，前身是哈军工", "地址": "哈尔滨市南岗区南通大街145号"},
    {"名称": "果戈里大街", "简介": "百年历史商业街，俄式风情浓郁", "地址": "哈尔滨市南岗区果戈里大街"},
    {"名称": "哈尔滨森林植物园", "简介": "中国最具代表性的寒温带植物园", "地址": "哈尔滨市香坊区哈平路105号"},
    {"名称": "哈尔滨文化公园", "简介": "东北地区最早的大型露天游乐园", "地址": "哈尔滨市南岗区南通大街208号"},
    {"名称": "哈尔滨国际会展中心", "简介": "黑龙江省最大的现代化展览中心", "地址": "哈尔滨市南岗区红旗大街301号"},
    {"名称": "哈尔滨啤酒博物馆", "简介": "展示百年哈尔滨啤酒历史的文化场馆", "地址": "哈尔滨市平房区哈啤路9号"},
    {"名称": "马迭尔宾馆", "简介": "百年历史俄式宾馆，中央大街标志性建筑", "地址": "哈尔滨市道里区中央大街89号"},
    {"名称": "华梅西餐厅", "简介": "哈尔滨四大西餐厅之首，百年俄式餐厅", "地址": "哈尔滨市道里区中央大街112号"},
    {"名称": "波特曼西餐厅", "简介": "著名俄式西餐厅，环境优雅", "地址": "哈尔滨市道里区中央大街西七道街"},
    {"名称": "老厨家", "简介": "百年老字号，锅包肉发源地", "地址": "哈尔滨市道里区友谊路318号"},
    {"名称": "张包铺", "简介": "百年老店，以排骨包子闻名", "地址": "哈尔滨市道外区南勋街与南二道街交口"},
    {"名称": "东方饺子王", "简介": "哈尔滨著名饺子连锁品牌", "地址": "哈尔滨市道里区中央大街39号"},
    {"名称": "山河屯铁锅炖", "简介": "东北特色铁锅炖代表餐厅", "地址": "哈尔滨市南岗区闽江路111号"},
    {"名称": "金刚山烧烤", "简介": "哈尔滨人气最旺的烧烤店", "地址": "哈尔滨市南岗区中山路252号"},
    {"名称": "秋林公司", "简介": "百年历史商场，红肠、大列巴原产地", "地址": "哈尔滨市南岗区东大直街319号"},
    {"名称": "哈尔滨融创乐园", "简介": "大型主题乐园，四季开放", "地址": "哈尔滨市松北区世茂大道99号"},
    {"名称": "黑龙江省广播电视塔", "简介": "即龙塔，哈尔滨城市地标", "地址": "哈尔滨市南岗区长江路178号"},
    {"名称": "哈尔滨规划展览馆", "简介": "展示哈尔滨城市规划的现代化展馆", "地址": "哈尔滨市道里区友谊路369号"},
    {"名称": "哈尔滨儿童公园", "简介": "中国首个儿童铁路主题公园", "地址": "哈尔滨市南岗区果戈里大街295号"},
    {"名称": "哈尔滨文庙", "简介": "东北地区最大孔庙", "地址": "哈尔滨市南岗区文庙街25号"},
    {"名称": "呼兰河口湿地公园", "简介": "松花江畔大型湿地公园", "地址": "哈尔滨市呼兰区呼口大桥东侧"},
    {"名称": "哈尔滨太平国际机场", "简介": "黑龙江省最大航空枢纽", "地址": "哈尔滨市道里区机场路1号"},
    {"名称": "哈尔滨火车站", "简介": "百年历史车站，欧式风格新站房", "地址": "哈尔滨市南岗区铁路街1号"},
    {"名称": "哈尔滨松花江公路大桥", "简介": "连接江南江北的主要通道", "地址": "哈尔滨市道里区松花江畔"}
]

# 生成CSV文件
with open('harbin_attractions.csv', 'w', newline='', encoding='utf-8-sig') as csvfile:
    fieldnames = ['名称', '简介', '地址']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    for place in places:
        writer.writerow(place)

print("CSV文件已生成：harbin_attractions.csv")

CSV文件已生成：harbin_attractions.csv
